In [0]:
from pyspark.sql.functions import col, sum as spark_sum, count, when, round as spark_round, date_format


In [0]:
accounts_df = spark.read.csv("/Volumes/azuredatabricks0811/default/silver/accounts", header=True, inferSchema=True)
sales_pipeline_df = spark.read.csv("/Volumes/azuredatabricks0811/default/silver/sales_pipeline", header=True, inferSchema=True)
products_df = spark.read.csv("/Volumes/azuredatabricks0811/default/silver/products", header=True, inferSchema=True)
sales_teams_df = spark.read.csv("/Volumes/azuredatabricks0811/default/silver/sales_teams", header=True, inferSchema=True)


## 3. Revenue by sector
Which industries generate the most account revenue.

In [0]:
gold_revenue_by_sector = (accounts_df
    .groupBy("sector")
    .agg(spark_round(spark_sum("revenue"), 2).alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)
gold_revenue_by_sector.show()


+------------------+-------------+
|            sector|total_revenue|
+------------------+-------------+
|          software|     30950.45|
|         technolgy|     27780.53|
|            retail|     27355.76|
|           medical|     16976.88|
|telecommunications|     16461.91|
|           finance|     16233.34|
|         marketing|     13070.38|
|     entertainment|      9665.22|
|        employment|      6104.64|
|          services|      4944.69|
+------------------+-------------+



## 4. Win rate by sales agent
Percentage of each agent's deals that were Won, out of Won + Lost (Prospecting/Engaging deals are still open, so excluded from win rate).

In [0]:
closed_deals_df = sales_pipeline_df.filter(col("deal_stage").isin(["Won", "Lost"]))

gold_win_rate_by_agent = (closed_deals_df
    .groupBy("sales_agent")
    .agg(
        count(when(col("deal_stage") == "Won", True)).alias("deals_won"),
        count(when(col("deal_stage") == "Lost", True)).alias("deals_lost"),
        count("*").alias("total_closed_deals")
    )
    .withColumn("win_rate_pct", spark_round((col("deals_won") / col("total_closed_deals")) * 100, 1))
    .orderBy(col("win_rate_pct").desc())
)
gold_win_rate_by_agent.show()


+------------------+---------+----------+------------------+------------+
|       sales_agent|deals_won|deals_lost|total_closed_deals|win_rate_pct|
+------------------+---------+----------+------------------+------------+
|     Hayden Neloms|      107|        45|               152|        70.4|
|   Maureen Marcano|      149|        64|               213|        70.0|
|    Wilburn Farren|       55|        24|                79|        69.6|
|    Cecily Lampkin|      107|        53|               160|        66.9|
| Versie Hillebrand|      176|        88|               264|        66.7|
|       Moses Frase|      129|        66|               195|        66.2|
|         Boris Faz|      101|        52|               153|        66.0|
|    James Ascencio|      135|        71|               206|        65.5|
|   Rosalina Dieter|       72|        38|               110|        65.5|
|     Corliss Cosme|      150|        79|               229|        65.5|
|      Reed Clapper|      155|        

## 5. Total sales by month
Sum of close_value for Won deals, grouped by the month they closed.

In [0]:
gold_sales_by_month = (sales_pipeline_df
    .filter(col("deal_stage") == "Won")
    .withColumn("close_month", date_format(col("close_date"), "yyyy-MM"))
    .groupBy("close_month")
    .agg(spark_round(spark_sum("close_value"), 2).alias("total_sales"))
    .orderBy("close_month")
)
gold_sales_by_month.show(20)


+-----------+-----------+
|close_month|total_sales|
+-----------+-----------+
|    2017-03|    1134672|
|    2017-04|     721932|
|    2017-05|    1025713|
|    2017-06|    1338466|
|    2017-07|     696932|
|    2017-08|    1050059|
|    2017-09|    1235264|
|    2017-10|     731980|
|    2017-11|     938943|
|    2017-12|    1131573|
+-----------+-----------+



## 6. Top products by deals won
Which products close (win) most often.

In [0]:
gold_top_products = (sales_pipeline_df
    .filter(col("deal_stage") == "Won")
    .groupBy("product")
    .agg(count("*").alias("deals_won"))
    .orderBy(col("deals_won").desc())
)
gold_top_products.show()


+--------------+---------+
|       product|deals_won|
+--------------+---------+
|     GTX Basic|      915|
|    MG Special|      793|
|        GTXPro|      729|
|   MG Advanced|      654|
|GTX Plus Basic|      653|
|  GTX Plus Pro|      479|
|       GTK 500|       15|
+--------------+---------+



## 7. Revenue by company office location
Note: this uses `office_location` from accounts (where the CLIENT company is based),
not `regional_office` from sales_teams (where our SALES AGENTS are based) - these are different things.

In [0]:
gold_revenue_by_location = (accounts_df
    .groupBy("office_location")
    .agg(spark_round(spark_sum("revenue"), 2).alias("total_revenue"))
    .orderBy(col("total_revenue").desc())
)
gold_revenue_by_location.show()


+---------------+-------------+
|office_location|total_revenue|
+---------------+-------------+
|  United States|    142997.85|
|          Korea|      8170.38|
|          Japan|      5158.71|
|         Jordan|      3027.46|
|         Panama|      2938.67|
|        Belgium|       1376.8|
|         Norway|      1223.72|
|        Germany|      1012.72|
|         Poland|       894.37|
|          Italy|       894.33|
|          Kenya|       647.18|
|     Philipines|       587.34|
|         Brazil|       405.59|
|        Romania|       167.89|
|          China|        40.79|
+---------------+-------------+



## 8. Write Gold tables

In [0]:
gold_revenue_by_sector.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/gold/revenue_by_sector")
gold_win_rate_by_agent.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/gold/win_rate_by_agent")
gold_sales_by_month.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/gold/sales_by_month")
gold_top_products.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/gold/top_products")
gold_revenue_by_location.write.option("header", "true").mode("overwrite").csv("/Volumes/azuredatabricks0811/default/gold/revenue_by_location")

print("All Gold tables written successfully.")


All Gold tables written successfully.
